# 🎴 LOR Clustering — Jerárquico, K-Means y GMM

**Objetivo:** descubrir arquetipos latentes en los mazos del leaderboard Maestro aplicando tres algoritmos de clustering con k=3 grupos.

**Pipeline:**
1. Carga y limpieza del dataset
2. Vectorización de mazos (cartas + facciones)
3. Escalado y reducción PCA
4. Clustering Jerárquico (Ward)
5. K-Means (k=3)
6. GMM — Gaussian Mixture Model (k=3)
7. Comparación y caracterización de clusters

> **Archivo requerido:** sube `lor_dataset.csv` antes de ejecutar.

---
## 📦 Celda 1 — Imports

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

warnings.filterwarnings('ignore')
np.random.seed(42)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 150,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
})

CLUSTER_COLORS = ['#E63946', '#457B9D', '#2A9D8F']
CLUSTER_NAMES  = ['Cluster 0', 'Cluster 1', 'Cluster 2']

print('✅ Imports OK')

---
## 📂 Celda 2 — Carga y limpieza

In [ ]:
df_raw = pd.read_csv('lor_dataset.csv')
df_raw['total_cards'] = pd.to_numeric(df_raw['total_cards'], errors='coerce')

# Eliminar mazos incompletos (< 40 cartas)
df = df_raw[df_raw['total_cards'] >= 40].copy().reset_index(drop=True)
df['rank'] = pd.to_numeric(df['rank'], errors='coerce')
df['lp']   = pd.to_numeric(df['lp'],   errors='coerce')
df['card_list_parsed'] = df['card_list'].apply(json.loads)

print(f'Filas originales  : {len(df_raw)}')
print(f'Eliminadas (<40c) : {len(df_raw) - len(df)}')
print(f'Mazos para análisis: {len(df)}')
display(df[['rank','player_name','lp','factions','total_cards']].head(5))

---
## 🔢 Celda 3 — Vectorización de mazos

Dos bloques de features:
- **Bloque A — Cartas** (N dims): copias de cada carta única presente en el dataset (0/1/2/3)
- **Bloque B — Facciones** (M dims): one-hot por facción

> Las columnas de maná están en NaN en el dataset actual (requieren Data Dragon), por lo que se excluyen. Los bloques A y B son suficientes para capturar la composición y el estilo del mazo.

In [ ]:
KNOWN_FACTIONS = [
    'Demacia', 'Freljord', 'Ionia', 'Noxus', 'Piltover & Zaun',
    'Shadow Isles', 'Bilgewater', 'Shurima', 'Mount Targon',
    'Bandle City', 'Runeterra',
]

# Universo de cartas y facciones
card_universe = sorted({c['c'] for cl in df['card_list_parsed'] for c in cl})
unknown_facs  = sorted({
    f.strip()
    for facs in df['factions']
    for f in facs.split('|')
    if f.strip() not in KNOWN_FACTIONS and f.strip()
})
all_factions = KNOWN_FACTIONS + unknown_facs

card_idx    = {c: i for i, c in enumerate(card_universe)}
faction_idx = {f: i for i, f in enumerate(all_factions)}
n_cards, n_fac = len(card_universe), len(all_factions)

matrix = np.zeros((len(df), n_cards + n_fac), dtype=np.float64)

for ri, (_, row) in enumerate(df.iterrows()):
    # Bloque A: copias de cartas
    for c in row['card_list_parsed']:
        col = card_idx.get(c['c'])
        if col is not None:
            matrix[ri, col] = c['n']
    # Bloque B: facciones one-hot
    for f in row['factions'].split('|'):
        col = faction_idx.get(f.strip())
        if col is not None:
            matrix[ri, n_cards + col] = 1.0

card_cols    = [f'card_{c}' for c in card_universe]
faction_cols = [f'fac_{f.replace(" ","_").replace("&","and")}' for f in all_factions]
feature_names = card_cols + faction_cols

print(f'✅ Vectorización completada')
print(f'   Bloque A — cartas   : {n_cards} dims')
print(f'   Bloque B — facciones: {n_fac} dims')
print(f'   Total               : {n_cards + n_fac} dims')
print(f'   Matriz              : {matrix.shape}')

---
## ⚖️ Celda 4 — Escalado y reducción PCA

In [ ]:
# Escalado
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(matrix)

# PCA completo para análisis de varianza
n_max    = min(len(X_scaled), X_scaled.shape[1])
pca_full = PCA(n_components=n_max, random_state=42)
pca_full.fit(X_scaled)
cumvar   = np.cumsum(pca_full.explained_variance_ratio_)
n_95     = int(np.searchsorted(cumvar, 0.95)) + 1

# PCA para clustering: retener 95% de varianza
pca_clust  = PCA(n_components=n_95, random_state=42)
X_pca      = pca_clust.fit_transform(X_scaled)

# PCA 2D para visualización
pca_2d     = PCA(n_components=2, random_state=42)
X_2d       = pca_2d.fit_transform(X_scaled)
var1, var2 = pca_2d.explained_variance_ratio_ * 100

print(f'✅ Escalado y PCA completados')
print(f'   Componentes para 95% varianza : {n_95}')
print(f'   Varianza PC1: {var1:.1f}%  |  PC2: {var2:.1f}%')
print(f'   X_pca shape : {X_pca.shape}  (entrada para clustering)')

# Gráfica de varianza acumulada
fig, ax = plt.subplots(figsize=(9, 4))
n_show = min(n_max, 40)
ax.plot(range(1, n_show+1), cumvar[:n_show]*100, 'o-',
        color='steelblue', linewidth=2, markersize=4)
ax.axhline(95, color='red',    linestyle='--', linewidth=1.2, label='95%')
ax.axvline(n_95, color='red',  linestyle=':', linewidth=1)
ax.annotate(f'n={n_95}', xy=(n_95, 95), xytext=(n_95+0.5, 91),
            fontsize=9, color='red')
ax.set_xlabel('Número de componentes PCA')
ax.set_ylabel('Varianza acumulada (%)')
ax.set_title('Varianza explicada acumulada — selección de componentes para clustering',
             fontweight='bold')
ax.legend()
ax.grid(alpha=0.4)
plt.tight_layout()
plt.savefig('lor_pca_varianza.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🌳 Celda 5 — Clustering Jerárquico (Ward)

El método Ward minimiza la varianza intracluster en cada fusión. El dendrograma muestra la estructura jerárquica completa y permite elegir el número de clusters de forma visual.

In [ ]:
K = 3   # número de clusters objetivo

# Linkage matrix
Z = linkage(X_pca, method='ward', metric='euclidean')

# ── Dendrograma ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))

# Colorear los k=3 clusters principales
from scipy.cluster.hierarchy import set_link_color_palette
set_link_color_palette(CLUSTER_COLORS)

# Calcular umbral de corte para k=3
last_merges   = Z[:, 2]  # distancias de fusión
cut_threshold = (last_merges[-K+1] + last_merges[-K]) / 2

dend = dendrogram(
    Z,
    labels=df['player_name'].values,
    leaf_rotation=90,
    leaf_font_size=7,
    color_threshold=cut_threshold,
    above_threshold_color='gray',
    ax=ax,
)
ax.axhline(cut_threshold, color='black', linestyle='--',
           linewidth=1.2, label=f'Corte k={K}  (dist={cut_threshold:.2f})')
ax.set_title(f'Dendrograma — Clustering Jerárquico Ward  (k={K})',
             fontweight='bold')
ax.set_xlabel('Jugador')
ax.set_ylabel('Distancia de fusión (Ward)')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('lor_dendrograma.png', dpi=150, bbox_inches='tight')
plt.show()

# Asignar etiquetas con k=3
labels_hier = fcluster(Z, K, criterion='maxclust') - 1  # 0-indexed
df['cluster_hier'] = labels_hier

sil_hier = silhouette_score(X_pca, labels_hier)
db_hier  = davies_bouldin_score(X_pca, labels_hier)
ch_hier  = calinski_harabasz_score(X_pca, labels_hier)

print(f'\nClustering Jerárquico  k={K}')
print(f'  Silhouette score      : {sil_hier:.4f}   (más alto = mejor, máx 1)')
print(f'  Davies-Bouldin score  : {db_hier:.4f}   (más bajo = mejor)')
print(f'  Calinski-Harabasz     : {ch_hier:.4f}   (más alto = mejor)')
print(f'\n  Distribución de clusters:')
print(pd.Series(labels_hier).value_counts().sort_index()
      .rename(index={0:'Cluster 0', 1:'Cluster 1', 2:'Cluster 2'}).to_string())

---
## 🔵 Celda 6 — K-Means (k=3)

Primero se evalúa el método del codo y el silhouette score para k=2..8, luego se entrena el modelo final con k=3.

In [ ]:
K_range = range(2, min(9, len(df)))
inertias, sil_scores = [], []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_pca)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_pca, lbl))

# ── Gráfica del codo + silhouette ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(list(K_range), inertias, 'o-', color='steelblue', linewidth=2)
axes[0].axvline(K, color='red', linestyle='--', linewidth=1.2, label=f'k={K}')
axes[0].set_xlabel('k (número de clusters)')
axes[0].set_ylabel('Inercia (WCSS)')
axes[0].set_title('Método del codo — K-Means', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.4)

axes[1].plot(list(K_range), sil_scores, 'o-', color='#2A9D8F', linewidth=2)
axes[1].axvline(K, color='red', linestyle='--', linewidth=1.2, label=f'k={K}')
axes[1].set_xlabel('k (número de clusters)')
axes[1].set_ylabel('Silhouette score')
axes[1].set_title('Silhouette score por k', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.4)

plt.tight_layout()
plt.savefig('lor_kmeans_codo.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Modelo final k=3 ──────────────────────────────────────────────────────────
km_final     = KMeans(n_clusters=K, random_state=42, n_init=20)
labels_km    = km_final.fit_predict(X_pca)
df['cluster_km'] = labels_km

sil_km = silhouette_score(X_pca, labels_km)
db_km  = davies_bouldin_score(X_pca, labels_km)
ch_km  = calinski_harabasz_score(X_pca, labels_km)

print(f'\nK-Means  k={K}')
print(f'  Silhouette score      : {sil_km:.4f}')
print(f'  Davies-Bouldin score  : {db_km:.4f}')
print(f'  Calinski-Harabasz     : {ch_km:.4f}')
print(f'  Inercia               : {km_final.inertia_:.2f}')
print(f'\n  Distribución de clusters:')
print(pd.Series(labels_km).value_counts().sort_index()
      .rename(index={0:'Cluster 0', 1:'Cluster 1', 2:'Cluster 2'}).to_string())

---
## 🟢 Celda 7 — GMM — Gaussian Mixture Model (k=3)

El GMM asigna probabilidades de pertenencia a cada cluster (soft assignment), lo que permite identificar mazos en zonas de transición entre arquetipos. Se evalúa BIC y AIC para k=2..8.

In [ ]:
bic_scores, aic_scores = [], []

for k in K_range:
    gmm = GaussianMixture(n_components=k, random_state=42, n_init=5)
    gmm.fit(X_pca)
    bic_scores.append(gmm.bic(X_pca))
    aic_scores.append(gmm.aic(X_pca))

# ── Gráfica BIC/AIC ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(K_range), bic_scores, 'o-', color='#E63946', linewidth=2, label='BIC')
ax.plot(list(K_range), aic_scores, 's-', color='#457B9D', linewidth=2, label='AIC')
ax.axvline(K, color='gray', linestyle='--', linewidth=1.2, label=f'k={K}')
ax.set_xlabel('k (número de componentes)')
ax.set_ylabel('Criterio de información')
ax.set_title('GMM — BIC y AIC por número de componentes', fontweight='bold')
ax.legend()
ax.grid(alpha=0.4)
plt.tight_layout()
plt.savefig('lor_gmm_bic.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'  k óptimo por BIC: {list(K_range)[np.argmin(bic_scores)]}')
print(f'  k óptimo por AIC: {list(K_range)[np.argmin(aic_scores)]}')

# ── Modelo final k=3 ──────────────────────────────────────────────────────────
gmm_final       = GaussianMixture(n_components=K, random_state=42, n_init=10, reg_covar=1e-6,
                                  covariance_type='full')
gmm_final.fit(X_pca)
labels_gmm      = gmm_final.predict(X_pca)
proba_gmm       = gmm_final.predict_proba(X_pca)   # probabilidades de pertenencia
df['cluster_gmm']  = labels_gmm
df['gmm_max_prob'] = proba_gmm.max(axis=1).round(4)

sil_gmm = silhouette_score(X_pca, labels_gmm)
db_gmm  = davies_bouldin_score(X_pca, labels_gmm)
ch_gmm  = calinski_harabasz_score(X_pca, labels_gmm)

print(f'\nGMM  k={K}  (covariance=full)')
print(f'  Silhouette score      : {sil_gmm:.4f}')
print(f'  Davies-Bouldin score  : {db_gmm:.4f}')
print(f'  Calinski-Harabasz     : {ch_gmm:.4f}')
print(f'  BIC                   : {gmm_final.bic(X_pca):.2f}')
print(f'  AIC                   : {gmm_final.aic(X_pca):.2f}')
print(f'\n  Distribución de clusters:')
print(pd.Series(labels_gmm).value_counts().sort_index()
      .rename(index={0:'Cluster 0', 1:'Cluster 1', 2:'Cluster 2'}).to_string())

print(f'\n  Probabilidad media de pertenencia (confianza del GMM):')
print(f'  {df["gmm_max_prob"].describe().round(4).to_string()}')

---
## 📊 Celda 8 — Visualización PCA 2D: los tres algoritmos

Los tres modelos proyectados en el mismo espacio PCA 2D para comparar visualmente cómo cada algoritmo divide el espacio.

In [ ]:
# También añadimos PC1/PC2 al dataframe para facilitar gráficas
df['PC1'] = X_2d[:, 0]
df['PC2'] = X_2d[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Clusters en espacio PCA 2D  (PC1={var1:.1f}%  PC2={var2:.1f}% varianza)',
             fontweight='bold', fontsize=14)

configs = [
    ('cluster_hier', 'Jerárquico Ward',  sil_hier, db_hier),
    ('cluster_km',   'K-Means',          sil_km,   db_km),
    ('cluster_gmm',  'GMM',              sil_gmm,  db_gmm),
]

for ax, (col, title, sil, db) in zip(axes, configs):
    labels = df[col].values
    for k in range(K):
        mask = labels == k
        ax.scatter(
            df.loc[mask, 'PC1'], df.loc[mask, 'PC2'],
            c=CLUSTER_COLORS[k], label=f'Cluster {k}  (n={mask.sum()})',
            s=80, alpha=0.85, edgecolors='white', linewidths=0.5, zorder=3,
        )

    # Centroide de cada cluster
    for k in range(K):
        mask = labels == k
        cx, cy = df.loc[mask, 'PC1'].mean(), df.loc[mask, 'PC2'].mean()
        ax.scatter(cx, cy, c=CLUSTER_COLORS[k], marker='*',
                   s=300, edgecolors='black', linewidths=0.8, zorder=5)

    # Etiquetas de nombre
    for _, row in df.iterrows():
        ax.annotate(row['player_name'],
                    xy=(row['PC1'], row['PC2']),
                    xytext=(3, 3), textcoords='offset points',
                    fontsize=5.5, alpha=0.65)

    ax.set_xlabel(f'PC1 ({var1:.1f}%)')
    ax.set_ylabel(f'PC2 ({var2:.1f}%)')
    ax.set_title(f'{title}\nSil={sil:.3f}  DB={db:.3f}', fontweight='bold')
    ax.legend(loc='upper right', framealpha=0.85)
    ax.axhline(0, color='gray', lw=0.4, ls='--')
    ax.axvline(0, color='gray', lw=0.4, ls='--')
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('lor_clusters_pca2d.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 📈 Celda 9 — Comparación de métricas entre algoritmos

In [ ]:
metrics_df = pd.DataFrame({
    'Algoritmo':         ['Jerárquico Ward', 'K-Means', 'GMM'],
    'Silhouette ↑':      [sil_hier, sil_km, sil_gmm],
    'Davies-Bouldin ↓':  [db_hier,  db_km,  db_gmm],
    'Calinski-Harabasz ↑': [ch_hier, ch_km, ch_gmm],
}).set_index('Algoritmo').round(4)

print('Comparación de métricas internas de clustering (k=3):')
display(metrics_df)

# Highlight mejor por métrica
best_sil = metrics_df['Silhouette ↑'].idxmax()
best_db  = metrics_df['Davies-Bouldin ↓'].idxmin()
best_ch  = metrics_df['Calinski-Harabasz ↑'].idxmax()
print(f'\n  Mejor Silhouette      : {best_sil}')
print(f'  Mejor Davies-Bouldin  : {best_db}')
print(f'  Mejor Calinski-Harabasz: {best_ch}')

# ── Gráfica de barras comparativa ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Comparación de métricas de calidad  (k=3)', fontweight='bold')

algs    = metrics_df.index.tolist()
alg_col = ['#E63946', '#457B9D', '#2A9D8F']

for ax, (metric, arrow) in zip(axes, [
    ('Silhouette ↑', '↑ mayor es mejor'),
    ('Davies-Bouldin ↓', '↓ menor es mejor'),
    ('Calinski-Harabasz ↑', '↑ mayor es mejor'),
]):
    vals = metrics_df[metric].values
    bars = ax.bar(algs, vals, color=alg_col, edgecolor='white', linewidth=0.5, alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')
    ax.set_title(f'{metric}\n{arrow}', fontsize=11)
    ax.set_xticklabels(algs, rotation=15, ha='right')
    ax.grid(axis='y', alpha=0.4)
    ax.set_ylim(0, max(vals) * 1.2)

plt.tight_layout()
plt.savefig('lor_metricas_comparacion.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🔍 Celda 10 — Caracterización de clusters

¿Qué define a cada cluster? Se analiza el modelo con mejor Silhouette score, mostrando las **facciones dominantes** y las **cartas más frecuentes** por cluster.

In [ ]:
# Usar el modelo con mejor Silhouette
scores = {'Jerárquico Ward': sil_hier, 'K-Means': sil_km, 'GMM': sil_gmm}
best_model = max(scores, key=scores.get)
col_map    = {'Jerárquico Ward': 'cluster_hier', 'K-Means': 'cluster_km', 'GMM': 'cluster_gmm'}
best_col   = col_map[best_model]

print(f'Modelo seleccionado para caracterización: {best_model} (Sil={scores[best_model]:.4f})\n')

FACTION_ID_MAP = {
    'DE': 'Demacia',      'FR': 'Freljord',        'IO': 'Ionia',
    'NX': 'Noxus',        'PZ': 'Piltover & Zaun', 'SI': 'Shadow Isles',
    'BW': 'Bilgewater',   'SH': 'Shurima',         'MT': 'Mount Targon',
    'BC': 'Bandle City',  'RU': 'Runeterra',
}

TOP_CARDS = 8

fig, axes = plt.subplots(K, 2, figsize=(14, 4*K))
fig.suptitle(f'Caracterización de clusters — {best_model}', fontweight='bold', fontsize=14)

for k in range(K):
    subset = df[df[best_col] == k]
    n_k    = len(subset)

    # ── Facciones más frecuentes en el cluster ────────────────────────────────
    fac_counts = {}
    for facs in subset['factions']:
        for f in facs.split('|'):
            f = f.strip()
            if f:
                fac_counts[f] = fac_counts.get(f, 0) + 1
    fac_series = pd.Series(fac_counts).sort_values(ascending=True)

    ax_fac = axes[k, 0]
    colors_fac = [CLUSTER_COLORS[k]] * len(fac_series)
    ax_fac.barh(fac_series.index, fac_series.values,
                color=colors_fac, alpha=0.8, edgecolor='white')
    ax_fac.set_title(f'Cluster {k}  (n={n_k}) — Facciones', fontweight='bold')
    ax_fac.set_xlabel('N.° de mazos con esta facción')
    ax_fac.grid(axis='x', alpha=0.4)
    for bar, val in zip(ax_fac.patches, fac_series.values):
        ax_fac.text(val + 0.1, bar.get_y() + bar.get_height()/2,
                    str(val), va='center', fontsize=8)

    # ── Cartas más frecuentes en el cluster ───────────────────────────────────
    card_counts = {}
    for cl in subset['card_list_parsed']:
        for c in cl:
            code = c['c']
            fac_key  = code[2:4] if len(code) >= 4 else ''
            fac_name = FACTION_ID_MAP.get(fac_key, fac_key)
            label    = f"{code}  [{fac_name}]"
            card_counts[label] = card_counts.get(label, 0) + 1
    top_cards = pd.Series(card_counts).sort_values(ascending=True).tail(TOP_CARDS)

    ax_card = axes[k, 1]
    ax_card.barh(top_cards.index, top_cards.values,
                 color=CLUSTER_COLORS[k], alpha=0.8, edgecolor='white')
    ax_card.set_title(f'Cluster {k}  (n={n_k}) — Top {TOP_CARDS} cartas más comunes',
                      fontweight='bold')
    ax_card.set_xlabel('N.° de mazos que la incluyen')
    ax_card.grid(axis='x', alpha=0.4)
    for bar, val in zip(ax_card.patches, top_cards.values):
        ax_card.text(val + 0.1, bar.get_y() + bar.get_height()/2,
                     str(val), va='center', fontsize=8)

plt.tight_layout()
plt.savefig('lor_cluster_caracterizacion.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🗺️ Celda 11 — Distribución de rank y LP por cluster

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Distribución de rank y LP por cluster (modelo seleccionado)',
             fontweight='bold')

# Boxplot rank por cluster
ax = axes[0]
data_rank = [df[df[best_col] == k]['rank'].values for k in range(K)]
bp = ax.boxplot(data_rank, labels=[f'C{k}' for k in range(K)],
                patch_artist=True,
                medianprops=dict(color='black', linewidth=1.5))
for patch, color in zip(bp['boxes'], CLUSTER_COLORS):
    patch.set_facecolor(color); patch.set_alpha(0.75)
for k, data in enumerate(data_rank):
    ax.scatter(np.random.normal(k+1, 0.05, len(data)), data,
               s=25, alpha=0.6, color=CLUSTER_COLORS[k],
               edgecolors='white', linewidths=0.4, zorder=3)
ax.set_title('Rank en el ladder', fontweight='bold')
ax.set_ylabel('Rank (menor = mejor)')
ax.grid(axis='y', alpha=0.4)

# Boxplot LP por cluster
ax2 = axes[1]
data_lp = [df[df[best_col] == k]['lp'].values for k in range(K)]
bp2 = ax2.boxplot(data_lp, labels=[f'C{k}' for k in range(K)],
                  patch_artist=True,
                  medianprops=dict(color='black', linewidth=1.5))
for patch, color in zip(bp2['boxes'], CLUSTER_COLORS):
    patch.set_facecolor(color); patch.set_alpha(0.75)
for k, data in enumerate(data_lp):
    ax2.scatter(np.random.normal(k+1, 0.05, len(data)), data,
                s=25, alpha=0.6, color=CLUSTER_COLORS[k],
                edgecolors='white', linewidths=0.4, zorder=3)
ax2.set_title('LP (League Points)', fontweight='bold')
ax2.set_ylabel('LP')
ax2.grid(axis='y', alpha=0.4)

# Tabla resumen de medias
ax3 = axes[2]
ax3.axis('off')
summary_data = []
for k in range(K):
    sub = df[df[best_col] == k]
    summary_data.append([
        f'Cluster {k}',
        len(sub),
        f"{sub['rank'].mean():.1f}",
        f"{sub['lp'].mean():.1f}",
        f"{sub['unique_cards'].mean():.1f}",
    ])
tbl = ax3.table(
    cellText=summary_data,
    colLabels=['Cluster', 'n', 'Rank\nmedio', 'LP\nmedio', 'Cartas\núnicas\nmedio'],
    cellLoc='center', loc='center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.2, 1.8)
for k in range(K):
    for j in range(5):
        tbl[k+1, j].set_facecolor(CLUSTER_COLORS[k] + '33')
ax3.set_title('Resumen estadístico por cluster', fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('lor_clusters_rank_lp.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 💾 Celda 12 — Exportar resultados

In [ ]:
# Exportar dataset con etiquetas de cluster asignadas
cols_export = ['extraction_date','rank','player_name','lp','factions',
               'total_cards','unique_cards','deck_code',
               'cluster_hier','cluster_km','cluster_gmm','gmm_max_prob',
               'PC1','PC2']
df[cols_export].to_csv('lor_dataset_clustered.csv', index=False)

# Guardar métricas
metrics_df.to_csv('lor_cluster_metrics.csv')

outputs = [
    'lor_dataset_clustered.csv',
    'lor_cluster_metrics.csv',
    'lor_pca_varianza.png',
    'lor_dendrograma.png',
    'lor_kmeans_codo.png',
    'lor_gmm_bic.png',
    'lor_clusters_pca2d.png',
    'lor_metricas_comparacion.png',
    'lor_cluster_caracterizacion.png',
    'lor_clusters_rank_lp.png',
]

print('✅ Archivos exportados:')
for f in outputs:
    print(f'   {f}')

# Descarga automática en Colab
import sys
if 'google.colab' in sys.modules:
    from google.colab import files
    import os
    for f in outputs:
        if os.path.exists(f):
            files.download(f)

print(f'\n  Modelo con mejor Silhouette: {best_model}  ({scores[best_model]:.4f})')
print(f'  Estos clusters son la variable objetivo para la siguiente etapa:')
print(f'  entrenamiento del clasificador supervisado (Random Forest / XGBoost).')